# LangSmith CI/CD Evaluation Demo

This notebook demonstrates a complete CI/CD pipeline that automatically evaluates your LangGraph agent whenever code changes are made.

## Overview

```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  Code Change    │────▶│  GitHub Action   │────▶│  LangSmith      │
│  (graph.py)     │     │  Triggers Eval   │     │  Evaluation     │
└─────────────────┘     └──────────────────┘     └────────┬────────┘
                                                          │
                                                          ▼
                        ┌──────────────────┐     ┌─────────────────-┐
                        │  PR Comment      │◀────│  Score Report    │
                        │  PASS / FAIL     │     │  (threshold 0.7) │
                        └──────────────────┘     └────────────────-─┘
```

**What you'll see:**
1. Multi-agent system implementation
2. Database tools for the agents
3. LangSmith evaluation setup
4. GitHub Actions workflow
5. Live demo: trigger an eval by opening a PR

---

# Part 1: The Multi-Agent System

Our agent uses a **supervisor pattern** with two specialized sub-agents:
- **Invoice Agent**: Handles billing and purchase queries
- **Music Agent**: Handles music catalog queries

The supervisor routes incoming questions to the appropriate agent.

In [1]:
# Display the agent implementation
with open("../app/graph.py", "r") as f:
    print(f.read())

"""Multi-agent system with supervisor routing to specialized subagents.

Version: 1.0.4 - Add status emojis to report
"""

from typing import Annotated
from typing_extensions import TypedDict, NotRequired

from langchain_openai import ChatOpenAI
from langgraph.graph.message import AnyMessage, add_messages
from langgraph.managed.is_last_step import RemainingSteps
from langgraph.prebuilt import create_react_agent
from langgraph_supervisor import create_supervisor

from app.tools import invoice_tools, music_tools


# State schema
class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    remaining_steps: NotRequired[RemainingSteps]


# Model
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)


# Invoice Subagent
invoice_prompt = """You are an invoice specialist for a digital music store.

Tools available:
- get_invoices_by_customer: Look up invoices by customer ID
- get_invoice_total: Get total spending for a customer

CRITICAL: Your response MUST include ALL 

## Agent Tools

The agents use tools that query the **Chinook sample database** (a music store database with customers, invoices, albums, and tracks).

In [2]:
# Display the tools implementation
with open("../app/tools.py", "r") as f:
    print(f.read())

"""Tools for the multi-agent system."""

import sqlite3
import requests
from langchain_core.tools import tool
from langchain_community.utilities.sql_database import SQLDatabase
from sqlalchemy import create_engine
from sqlalchemy.pool import StaticPool


def get_engine_for_chinook_db():
    """Load Chinook sample database into memory."""
    url = "https://raw.githubusercontent.com/lerocha/chinook-database/master/ChinookDatabase/DataSources/Chinook_Sqlite.sql"
    response = requests.get(url)
    sql_script = response.text

    connection = sqlite3.connect(":memory:", check_same_thread=False)
    connection.executescript(sql_script)
    return create_engine(
        "sqlite://",
        creator=lambda: connection,
        poolclass=StaticPool,
        connect_args={"check_same_thread": False},
    )


engine = get_engine_for_chinook_db()
db = SQLDatabase(engine)


# Invoice Tools
@tool
def get_invoices_by_customer(customer_id: str) -> str:
    """Get all invoices for a customer, sorted

## Try the Agent

Let's run a quick test to see the agent in action.

In [3]:
import sys
sys.path.insert(0, "..")

from dotenv import load_dotenv
load_dotenv("../.env")

from app import run_graph

# Test query
result = await run_graph({
    "messages": [{"role": "user", "content": "What albums does AC/DC have?"}]
})

print("Response:", result["output"])

Task supervisor with path ('__pregel_pull', 'supervisor') wrote to unknown channel remaining_steps, ignoring it.


Response: AC/DC has the following albums:
1. **For Those About To Rock We Salute You**
2. **Let There Be Rock**


---

# Part 2: The Evaluation System

The evaluation uses:
- **LangSmith Dataset**: Test cases with expected answers
- **LLM-as-Judge**: GPT-4o-mini evaluates semantic correctness
- **Threshold**: 70% score required to pass

In [4]:
# Display the evaluation test
with open("../tests/test_eval.py", "r") as f:
    print(f.read())

"""
LangSmith evaluation test for CI/CD.

This test runs the multi-agent system against a LangSmith dataset and evaluates
the results using LLM-as-judge. Results are saved to a config file that the
report script uses to generate PR comments.

Usage:
    pytest -m evaluator
"""

import json
import pytest
from dotenv import load_dotenv

load_dotenv()

from langchain_openai import ChatOpenAI
from langsmith import Client
from openevals.llm import create_async_llm_as_judge

from app.graph import run_graph

# Initialize LangSmith client
client = Client()

# Custom prompt for semantic correctness (less strict on wording)
# Uses OpenEvals placeholders: {inputs}, {outputs}, {reference_outputs}
SEMANTIC_CORRECTNESS_PROMPT = """You are evaluating whether an AI response contains the key facts from the expected answer.

<User Question>
{inputs}
</User Question>

<AI Response>
{outputs}
</AI Response>

<Expected Answer>
{reference_outputs}
</Expected Answer>

Evaluate based on SEMANTIC CORRECTNESS:


## Report Generation

After evals run, a report script queries LangSmith for results and generates a markdown summary.

In [5]:
# Display the report script
with open("../scripts/report_eval.py", "r") as f:
    print(f.read())

#!/usr/bin/env python3
"""
Generate evaluation report for PR comments.

Reads evaluation config files generated by test_eval.py, queries LangSmith
for results, and outputs a markdown summary suitable for PR comments.

Usage:
    python scripts/report_eval.py
    python scripts/report_eval.py evaluation_config__*.json
    python scripts/report_eval.py --output custom_report.md
"""

import argparse
import glob
import json
import operator
import os
import sys
from collections import defaultdict

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass  # dotenv not needed in CI where env vars are set directly

from langsmith import Client

# Operator map for threshold comparisons
OP_MAP = {
    ">": operator.gt,
    "<": operator.lt,
    ">=": operator.ge,
    "<=": operator.le,
    "==": operator.eq,
    "!=": operator.ne,
}


def parse_threshold(threshold_str: str) -> tuple:
    """Parse threshold expression like '>=0.75' into (operator, value)."""
    for 

---

# Part 3: GitHub Actions Workflow

The workflow:
1. **Triggers** on PRs that modify `app/graph.py` or `app/tools.py`
2. **Runs** the evaluation test via pytest
3. **Posts** results as a PR comment

In [6]:
# Display the GitHub Actions workflow
with open("../.github/workflows/evaluate.yml", "r") as f:
    print(f.read())

name: LangSmith Evaluation

on:
  pull_request:
    branches: [main]
    paths:
      - "app/graph.py"
      - "app/tools.py"

permissions:
  contents: read
  pull-requests: write

jobs:
  evaluate:
    runs-on: ubuntu-latest
    environment: production  # Use GitHub environment for secrets
    env:
      OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}
      LANGSMITH_API_KEY: ${{ secrets.LANGSMITH_API_KEY }}
      LANGSMITH_TRACING: "true"

    steps:
      - uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.12"

      - name: Install uv
        uses: astral-sh/setup-uv@v4

      - name: Install dependencies
        run: uv sync

      - name: Run evaluations
        run: uv run pytest -m evaluator --junitxml=results.xml

      - name: Upload evaluation configs
        uses: actions/upload-artifact@v4
        with:
          name: eval-configs
          path: evaluation_config__*.json

  report:
    ne

---

# Part 4: Trigger an Eval (Live Demo)

Now let's trigger an actual evaluation by:
1. Making a code change to the agent
2. Creating a branch and PR
3. Watching the GitHub Action run and post results

**Prerequisites:**
- `gh` CLI installed and authenticated (`gh auth login`)
- Git configured with push access to the repo

In [7]:
import subprocess
import uuid
import re
from datetime import datetime

def run(cmd: str) -> str:
    """Run a shell command and return output."""
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd="..")
    if result.returncode != 0:
        print(f"Error: {result.stderr}")
        raise Exception(result.stderr)
    return result.stdout.strip()

In [8]:
# Configuration
BRANCH_NAME = f"eval-test-{uuid.uuid4().hex[:8]}"
AGENT_FILE = "app/graph.py"

print(f"Branch name: {BRANCH_NAME}")

Branch name: eval-test-045e91c0


In [9]:
# Step 1: Ensure we're on main and up to date
print(run("git checkout main"))
print(run("git pull origin main"))

M	notebooks/trigger_eval.ipynb
M	pyproject.toml
Your branch is up to date with 'origin/main'.
Already up to date.


In [10]:
# Step 2: Create a new branch
print(run(f"git checkout -b {BRANCH_NAME}"))

In [11]:
# Step 3: Update the version string in the agent file
with open(f"../{AGENT_FILE}", "r") as f:
    content = f.read()

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
new_version = f"Version: notebook-triggered @ {timestamp}"

updated_content = re.sub(r'Version:.*', new_version, content)

with open(f"../{AGENT_FILE}", "w") as f:
    f.write(updated_content)

print(f"Updated {AGENT_FILE} with: {new_version}")

Updated app/graph.py with: Version: notebook-triggered @ 2026-02-18 22:13


In [12]:
# Step 4: Commit the change
run(f"git add {AGENT_FILE}")
print(run('git commit -m "test: trigger eval from notebook"'))

[eval-test-045e91c0 a87538a] test: trigger eval from notebook
 1 file changed, 1 insertion(+), 1 deletion(-)


In [13]:
# Step 5: Push the branch
print(run(f"git push -u origin {BRANCH_NAME}"))

branch 'eval-test-045e91c0' set up to track 'origin/eval-test-045e91c0'.


In [14]:
# Step 6: Create a PR
pr_title = "test: notebook-triggered eval"
pr_body = f"""## Summary
Automated PR created from Jupyter notebook to trigger eval workflow.

**Timestamp:** {timestamp}
**Branch:** {BRANCH_NAME}
"""

pr_url = run(f'gh pr create --title "{pr_title}" --body "{pr_body}"')
print(f"\n{'='*50}")
print(f"PR Created: {pr_url}")
print(f"{'='*50}")


PR Created: https://github.com/xuro-langchain/evals-cicd/pull/3


In [15]:
# Step 7: Open PR in browser to watch the workflow
print("Opening PR in browser...")
run("gh pr view --web")

Opening PR in browser...


''

---

## Cleanup

After watching the eval complete, run these cells to clean up.

In [ ]:
# Close the PR and delete the branch
print(run(f"gh pr close {BRANCH_NAME} --delete-branch"))

In [ ]:
# Switch back to main
print(run("git checkout main"))

---

# Summary

You've seen:

1. **Multi-Agent System** (`app/graph.py`)
   - Supervisor pattern with invoice and music agents
   - LangGraph for orchestration

2. **Database Tools** (`app/tools.py`)
   - SQLAlchemy + Chinook database
   - Invoice and music catalog queries

3. **LangSmith Evaluation** (`tests/test_eval.py`)
   - Dataset-driven testing
   - LLM-as-judge for semantic correctness
   - 70% threshold for passing

4. **CI/CD Pipeline** (`.github/workflows/evaluate.yml`)
   - Triggers on agent code changes
   - Runs evals automatically
   - Posts results as PR comments

This pattern ensures your agent quality is validated on every change!